# 005 — Error Analysis on Labeled Soundscapes

Run exp 012 ONNX models on 66 labeled soundscape files to identify where we lose.

- Per-species AUROC (75/234 species covered)
- Per-taxon breakdown (birds vs amphibians vs insects)
- Frequency range analysis (mel fmax per model)
- Confidence calibration and false positive patterns

In [ ]:
import json
from pathlib import Path

import librosa
import numpy as np
import onnxruntime as ort
import pandas as pd
import soundfile as sf
from sklearn.metrics import roc_auc_score

DATA_DIR = Path("/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026")
WEIGHTS_DIR = Path(
    "/storage/project/ps-clef2026_birdclef-0/mgustineli3/birdclef-2026"
    "/user/murilo/experiments/010-submission-pipeline/artifacts/weights"
)
SOUNDSCAPE_DIR = DATA_DIR / "train_soundscapes"
LABELS_CSV = DATA_DIR / "train_soundscapes_labels.csv"
TAXONOMY_CSV = DATA_DIR / "taxonomy.csv"

## Load ground truth + taxonomy

In [ ]:
taxonomy = pd.read_csv(TAXONOMY_CSV)
all_species = sorted(taxonomy["primary_label"].astype(str).tolist())
sp2idx = {sp: i for i, sp in enumerate(all_species)}
sp2taxon = dict(zip(taxonomy["primary_label"].astype(str), taxonomy["class_name"], strict=False))
print(f"Taxonomy: {len(all_species)} species, {taxonomy['class_name'].value_counts().to_dict()}")

# Parse labels -> per-segment multi-hot (filename, end_sec) -> set of species
labels_df = pd.read_csv(LABELS_CSV)
segments = {}  # (filename, end_sec) -> set of species codes
for _, row in labels_df.iterrows():
    fn = row["filename"]
    parts = row["end"].split(":")
    end_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    key = (fn, end_sec)
    if key not in segments:
        segments[key] = set()
    for sp in str(row["primary_label"]).split(";"):
        segments[key].add(sp.strip())

# Species present in labeled soundscapes
labeled_species = set()
for spp in segments.values():
    labeled_species |= spp
labeled_species = sorted(labeled_species & set(all_species))
print(
    f"Labeled segments: {len(segments)}, species covered: {len(labeled_species)}/{len(all_species)}"
)

# Build ground truth matrix: rows = segments, cols = all 234 species
seg_keys = sorted(segments.keys())
y_true = np.zeros((len(seg_keys), len(all_species)), dtype=np.float32)
for i, key in enumerate(seg_keys):
    for sp in segments[key]:
        if sp in sp2idx:
            y_true[i, sp2idx[sp]] = 1.0

print(f"GT matrix: {y_true.shape}, mean labels/segment: {y_true.sum(1).mean():.1f}")

## Inference helpers (reuse from submission.py)

In [ ]:
def sigmoid(x):
    return np.where(x >= 0, 1 / (1 + np.exp(-x)), np.exp(x) / (1 + np.exp(x)))


def compute_mel(audio, cfg):
    mel_type = cfg.get("type", "mel")
    if mel_type == "protoclr_mel":
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=cfg["sr"],
            n_fft=cfg["n_fft"],
            hop_length=cfg["hop_length"],
            n_mels=cfg["n_mels"],
            fmin=cfg.get("fmin", 50),
            fmax=cfg.get("fmax", 8000),
        )
        mel_db = 10.0 * np.log10(np.maximum(mel, 1e-10))
        mel_norm = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        mel_std = (mel_norm - cfg.get("protoclr_mean", 0.5347)) / cfg.get("protoclr_std", 0.0772)
        target = cfg.get("target_frames", 250)
        T = mel_std.shape[1]
        if T > target:
            mel_std = mel_std[:, :target]
        elif T < target:
            mel_std = np.pad(mel_std, ((0, 0), (0, target - T)))
        return mel_std[np.newaxis, np.newaxis].astype(np.float32)

    elif mel_type == "kaldi_fbank":
        import torch
        import torchaudio

        waveform = torch.from_numpy(audio).float()
        waveform = waveform - waveform.mean()
        mel = torchaudio.compliance.kaldi.fbank(
            waveform.unsqueeze(0),
            htk_compat=True,
            sample_frequency=cfg["sr"],
            use_energy=False,
            window_type="hanning",
            num_mel_bins=cfg["num_mel_bins"],
            dither=0.0,
            frame_shift=cfg["frame_shift"],
        )
        target = cfg.get("target_length", 512)
        if mel.shape[0] < target:
            mel = torch.nn.functional.pad(mel, (0, 0, 0, target - mel.shape[0]))
        elif mel.shape[0] > target:
            mel = mel[:target]
        mel = (mel - cfg["norm_mean"]) / (cfg["norm_std"] * 2)
        return mel.unsqueeze(0).unsqueeze(0).numpy().astype(np.float32)

    else:  # mel (ConvNeXt)
        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=cfg["sr"],
            n_fft=cfg["n_fft"],
            hop_length=cfg["hop_length"],
            n_mels=cfg["n_mels"],
            fmin=cfg.get("fmin", 0),
            fmax=cfg.get("fmax"),
            htk=cfg.get("htk", True),
        )
        mel = np.log(mel + 1e-6)
        mel = (mel - cfg["norm_mean"]) / (cfg["norm_std"] * 2)
        target = cfg.get("target_frames", 500)
        T = mel.shape[1]
        if T > target:
            s = (T - target) // 2
            mel = mel[:, s : s + target]
        elif T < target:
            mel = np.pad(mel, ((0, 0), (0, target - T)))
        return mel[np.newaxis, np.newaxis].astype(np.float32)


def load_audio_chunk(path, sr, start_s, dur=5.0):
    info = sf.info(path)
    audio, _ = sf.read(
        path,
        dtype="float32",
        start=int(start_s * info.samplerate),
        frames=int(dur * info.samplerate) + 2,
    )
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if info.samplerate != sr:
        audio = librosa.resample(audio, orig_sr=info.samplerate, target_sr=sr)
    target_samples = int(sr * dur)
    if len(audio) < target_samples:
        audio = np.pad(audio, (0, target_samples - len(audio)))
    return audio[:target_samples].astype(np.float32)

## Run inference with all 3 models

In [ ]:
MODELS = {
    "ProtoCLR": "protoclr-2026-soundscape",
    "ConvNeXt": "convnext-2026-soundscape",
    "EAT": "eat-2026-soundscape",
}

# {model_name: (n_segments, 234) prediction matrix}
y_pred = {}

for name, slug in MODELS.items():
    model_dir = WEIGHTS_DIR / slug
    with open(model_dir / "mel_config.json") as f:
        mel_cfg = json.load(f)
    with open(model_dir / "species_list.json") as f:
        model_species = json.load(f)
    with open(model_dir / "species_mapping.json") as f:
        sp_map = json.load(f)

    idx_map = sp_map.get("idx_to_name_2026") or sp_map["idx_2025_to_name_2026"]

    session = ort.InferenceSession(str(model_dir / "model.onnx"))
    input_name = session.get_inputs()[0].name
    sr = mel_cfg["sr"]

    preds = np.zeros((len(seg_keys), len(all_species)), dtype=np.float32)

    # Group segments by file for efficient audio loading
    from collections import defaultdict

    file_segs = defaultdict(list)
    for i, (fn, end_sec) in enumerate(seg_keys):
        file_segs[fn].append((i, end_sec))

    for fi, (fn, segs) in enumerate(sorted(file_segs.items())):
        audio_path = str(SOUNDSCAPE_DIR / fn)
        for seg_idx, end_sec in segs:
            start_s = end_sec - 5
            audio = load_audio_chunk(audio_path, sr, start_s)
            mel = compute_mel(audio, mel_cfg)
            logits = session.run(None, {input_name: mel})[0].flatten()
            probs = sigmoid(logits)
            # Map model outputs to 234-species vector
            for model_idx_str, sp_name in idx_map.items():
                if sp_name in sp2idx:
                    preds[seg_idx, sp2idx[sp_name]] = probs[int(model_idx_str)]

        if (fi + 1) % 10 == 0:
            print(f"  {name}: {fi+1}/{len(file_segs)} files")

    y_pred[name] = preds
    print(f"{name} done: {preds.shape}")

# Ensemble: simple mean
y_pred["Ensemble"] = np.mean([y_pred[n] for n in MODELS], axis=0)
print("\nAll models + ensemble complete.")

## Per-species AUROC (only species with ≥1 positive in labeled soundscapes)

In [ ]:
results = []
for sp_idx, sp in enumerate(all_species):
    col = y_true[:, sp_idx]
    n_pos = int(col.sum())
    if n_pos == 0 or n_pos == len(col):
        continue  # can't compute AUROC
    row = {"species": sp, "n_pos": n_pos, "taxon": sp2taxon.get(sp, "Unknown")}
    for model_name in list(MODELS.keys()) + ["Ensemble"]:
        row[f"auroc_{model_name}"] = roc_auc_score(col, y_pred[model_name][:, sp_idx])
    results.append(row)

results_df = pd.DataFrame(results).sort_values("auroc_Ensemble", ascending=True)
print(f"Species with AUROC computable: {len(results_df)}/{len(labeled_species)}")
print(f"\nMacro AUROC (on {len(results_df)} species):")
for m in list(MODELS.keys()) + ["Ensemble"]:
    col = f"auroc_{m}"
    print(f"  {m}: {results_df[col].mean():.4f}")

## Per-taxon breakdown

In [ ]:
for taxon in sorted(results_df["taxon"].unique()):
    sub = results_df[results_df["taxon"] == taxon]
    print(f"\n{taxon} ({len(sub)} species):")
    for m in list(MODELS.keys()) + ["Ensemble"]:
        col = f"auroc_{m}"
        print(f"  {m}: {sub[col].mean():.4f}  (min={sub[col].min():.3f}, max={sub[col].max():.3f})")

## Worst and best species

In [ ]:
print("=== 15 WORST species (Ensemble AUROC) ===")
worst = results_df.head(15)[
    ["species", "taxon", "n_pos", "auroc_Ensemble", "auroc_ProtoCLR", "auroc_ConvNeXt", "auroc_EAT"]
]
print(worst.to_string(index=False))

print("\n=== 15 BEST species (Ensemble AUROC) ===")
best = results_df.tail(15)[
    ["species", "taxon", "n_pos", "auroc_Ensemble", "auroc_ProtoCLR", "auroc_ConvNeXt", "auroc_EAT"]
]
print(best.to_string(index=False))

## Species coverage gap

159/234 species have NO labeled soundscape segments — we can't measure their performance here.
Check how many of those are in our training data at all.

In [ ]:
uncovered = sorted(set(all_species) - set(labeled_species))
uncovered_taxa = pd.Series([sp2taxon.get(sp, "Unknown") for sp in uncovered]).value_counts()
print(f"Species NOT in labeled soundscapes: {len(uncovered)}/234")
print(f"By taxon: {uncovered_taxa.to_dict()}")
print(f"\nSpecies IN labeled soundscapes: {len(labeled_species)}/234")
labeled_taxa = pd.Series([sp2taxon.get(sp, "Unknown") for sp in labeled_species]).value_counts()
print(f"By taxon: {labeled_taxa.to_dict()}")

## Confidence analysis: mean prediction on positive vs negative segments

In [ ]:
for model_name in list(MODELS.keys()) + ["Ensemble"]:
    p = y_pred[model_name]
    pos_mask = y_true == 1
    neg_mask = y_true == 0
    # Only on species present in labeled data
    sp_idxs = [sp2idx[sp] for sp in labeled_species]
    pos_vals = p[:, sp_idxs][pos_mask[:, sp_idxs]]
    neg_vals = p[:, sp_idxs][neg_mask[:, sp_idxs]]
    print(f"{model_name}:")
    print(f"  Positive segments: mean={pos_vals.mean():.4f}, median={np.median(pos_vals):.4f}")
    print(f"  Negative segments: mean={neg_vals.mean():.4f}, median={np.median(neg_vals):.4f}")
    print(f"  Separation ratio: {pos_vals.mean() / (neg_vals.mean() + 1e-8):.1f}x")

## Mel frequency coverage by model

In [ ]:
print("Model frequency coverage:")
for name, slug in MODELS.items():
    with open(WEIGHTS_DIR / slug / "mel_config.json") as f:
        cfg = json.load(f)
    fmax = cfg.get("fmax", "Nyquist")
    sr = cfg["sr"]
    nyquist = sr // 2
    if fmax is None or fmax == "Nyquist":
        fmax = nyquist
    print(f"  {name}: sr={sr}Hz, fmax={fmax}Hz (Nyquist={nyquist}Hz)")

print("\nInsect sonotypes typically have energy up to 20kHz+.")
print("ProtoCLR caps at 8kHz — may miss high-frequency insects entirely.")

## Ensemble vs single model: where does averaging help?

In [ ]:
results_df["ensemble_gain"] = results_df["auroc_Ensemble"] - results_df[
    ["auroc_ProtoCLR", "auroc_ConvNeXt", "auroc_EAT"]
].max(axis=1)

helped = results_df[results_df["ensemble_gain"] > 0.01]
hurt = results_df[results_df["ensemble_gain"] < -0.01]
print(f"Ensemble helps (>0.01): {len(helped)} species")
print(f"Ensemble hurts (<-0.01): {len(hurt)} species")
print(f"Neutral: {len(results_df) - len(helped) - len(hurt)} species")
print(f"\nMean ensemble gain: {results_df['ensemble_gain'].mean():.4f}")

if len(hurt) > 0:
    print("\nSpecies where ensemble hurts most:")
    print(
        hurt.sort_values("ensemble_gain")[["species", "taxon", "ensemble_gain", "auroc_Ensemble"]]
        .head(10)
        .to_string(index=False)
    )

## Summary

In [ ]:
print("=" * 60)
print("ERROR ANALYSIS SUMMARY")
print("=" * 60)
print(f"Evaluated on: {len(seg_keys)} segments, {len(file_segs)} files")
print(
    f"Species covered: {len(results_df)}/{len(all_species)} (only {len(labeled_species)} in labeled soundscapes)"
)
print()
print("Macro AUROC (labeled species only):")
for m in list(MODELS.keys()) + ["Ensemble"]:
    print(f"  {m}: {results_df[f'auroc_{m}'].mean():.4f}")
print()
print("Per-taxon Ensemble AUROC:")
for taxon in sorted(results_df["taxon"].unique()):
    sub = results_df[results_df["taxon"] == taxon]
    print(f"  {taxon} ({len(sub)} sp): {sub['auroc_Ensemble'].mean():.4f}")
print()
print(f"Species not evaluable (no labeled soundscape data): {len(uncovered)}")
print(f"  By taxon: {uncovered_taxa.to_dict()}")